# The Voice-To-Text Model: Everything You need to know

Welcome to Automatic Speech Recognition (ASR). In this tutorial, we will build a production-ready transcription pipeline using **Whisper**, a robust Seq2Seq Transformer model.

### The Analytical Architecture of Whisper:
1. **The Feature Extractor:** Takes a raw audio array sampled at exactly 16,000 Hz and mathematically computes an 80-channel log-Mel spectrogram.
2. **The Encoder:** A Transformer that processes the spectrogram and outputs a dense vector representation of the audio's semantic meaning and acoustic features.
3. **The Decoder:** An autoregressive Transformer that takes the Encoder's output and predicts the corresponding text tokens one by one.

In [ ]:
# Cell 2: Environment Setup and Imports
# In a real environment, you must install: !pip install -q transformers datasets torch librosa soundfile

import torch
import librosa
import numpy as np
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset

# Set device for tensor operations (GPU if available, otherwise CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print("ASR libraries imported successfully.")

## Step 1: Loading the AI Architecture

To build our pipeline, we must instantiate two coupled components from Hugging Face:
1. `WhisperProcessor`: This handles both the front-end (converting raw audio waveforms into Mel-Spectrogram tensors) and the back-end (converting predicted integer Token IDs back into readable English strings).
2. `WhisperForConditionalGeneration`: The massive neural network containing both the acoustic Encoder and the text Decoder. We will use the `whisper-tiny.en` variant for speed and efficiency.

In [ ]:
# Cell 4: Loading the Processor and Model

model_id = "openai/whisper-tiny.en"

print("Downloading/Loading the Whisper Processor...")
processor = WhisperProcessor.from_pretrained(model_id)

print("Downloading/Loading the Whisper Neural Network...")
model = WhisperForConditionalGeneration.from_pretrained(model_id).to(device)

# Force the model into evaluation mode (disables dropout layers for deterministic inference)
model.eval()

print("\nAnalytical Note: Whisper architecture is successfully loaded into memory.")

## Step 2: Data Ingestion and Sampling Rate Math

Neural networks are mathematically strict. Whisper was trained exclusively on audio sampled at **16,000 Hz** (16kHz). 

If you feed it an audio file sampled at 44.1kHz (CD quality) or 8kHz (Telephone quality), the frequency calculations in the spectrogram will be completely wrong, and the model will output gibberish. We must load our audio and resample it to the correct target rate. We will use a sample audio file from a public dataset.

In [ ]:
# Cell 6: Preparing the Audio Data

# 1. Load a sample dataset containing audio recordings of people talking
print("Loading sample audio dataset...")
dataset = load_dataset("PolyAI/minds14", name="en-US", split="train")

# 2. Extract the first audio sample
# The dataset object contains the raw array and the original sampling rate
sample = dataset[0]["audio"]
raw_audio_array = sample["array"]
original_sampling_rate = sample["sampling_rate"]

print(f"\nOriginal Audio Shape: {raw_audio_array.shape}")
print(f"Original Sampling Rate: {original_sampling_rate} Hz")

# 3. Resample the audio to exactly 16,000 Hz if necessary
TARGET_RATE = 16000

if original_sampling_rate != TARGET_RATE:
    print(f"Resampling audio from {original_sampling_rate}Hz to {TARGET_RATE}Hz...")
    # librosa performs the mathematical interpolation required to change the sample rate
    audio_array = librosa.resample(
        raw_audio_array, 
        orig_sr=original_sampling_rate, 
        target_sr=TARGET_RATE
    )
else:
    audio_array = raw_audio_array

print(f"Final Audio Array Shape for Model: {audio_array.shape}")

## Step 3: Feature Extraction (Waveform to Spectrogram)

Now we pass our 1D audio array to the `WhisperProcessor`. 

Under the hood, the processor applies a Short-Time Fourier Transform (STFT) to slice the audio into tiny 25-millisecond windows. It calculates the frequency magnitudes in each window, applies a Mel-scale filter (which mimics how the human ear perceives pitch), and compresses it into a log scale. The result is a 2D tensor that the neural network can "see".

In [ ]:
# Cell 8: Generating the Mel-Spectrogram

# Process the raw audio array into input features
inputs = processor(
    audio_array, 
    sampling_rate=TARGET_RATE, 
    return_tensors="pt" # Return PyTorch tensors
)

# Extract the calculated spectrogram and move it to our processing device (CPU/GPU)
input_features = inputs.input_features.to(device)

print("--- Analytical Feature Extraction ---")
print(f"Spectrogram Tensor Shape: {input_features.shape}")
print("Format: [Batch Size, Feature Channels (Mel Bins), Time Frames]")
print("Notice how the 1D audio array has been mathematically expanded into a 2D matrix!")

## Step 4: The Forward Pass and Sequence Decoding

This is the inference step. We pass the spectrogram into the `generate()` function. 

Whisper is an **Autoregressive Decoder**. It looks at the encoded audio and predicts the very first word. Then, it takes that predicted word, feeds it back into itself, and uses it to help predict the *second* word, looping until it predicts a special `<|endoftext|>` token.

In [ ]:
# Cell 10: Model Generation and Decoding

# Disable gradient tracking since we are just doing inference, saving memory
with torch.no_grad():
    print("Executing Neural Network Forward Pass...")
    # The model predicts the sequence of token IDs
    predicted_ids = model.generate(input_features)

print(f"\nRaw Predicted Token IDs:\n{predicted_ids[0]}")

# Decode the integer IDs back into human-readable text
# skip_special_tokens=True removes the <|startoftranscript|> and <|endoftext|> tags
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print("\n--- Final Transcription Result ---")
print(f"\"{transcription.strip()}\"")

## Conclusion

You have successfully built an Automatic Speech Recognition pipeline!

**Key Analytical Takeaways:**
1. **Audio is just Time-Series Data:** The raw sound wave is just a massive 1D array of floats representing air pressure over time.
2. **The Spectrogram Bridge:** Neural networks struggle with raw oscillating waveforms. The mathematical conversion to a Mel-Spectrogram transforms a confusing audio problem into a highly structured 2D Computer Vision problem.
3. **Seq2Seq Power:** Unlike older models that tried to predict characters frame-by-frame and required external dictionaries to spell words correctly, Whisper's Transformer Decoder learns the linguistic rules of English directly, outputting perfectly spelled and punctuated sentences entirely on its own!